# 02 – Data Preparation: Credit Card Fraud Detection

**Föregående steg:** `01_eda.ipynb` identifierade 1081 dubblettrader, extrem klassobalans (99,83 % / 0,17 %), och att `Time`/`Amount` behöver skalas separat.

**Den här notebooken:**
1. Tar bort dubbletter
2. Delar upp data i train/test (stratifierat på `Class`)
3. Skalar `Time` och `Amount` (fit endast på träningsdata)
4. Extraherar enbart legitima transaktioner för autoencoder-träning, och delar ut ett validation-set från dem
5. Sparar alla splits + scaler till disk för återanvändning i `03_model_training.ipynb`

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

RANDOM_STATE = 42


## Ladda data och ta bort dubbletter

EDA:n hittade 1081 dubblettrader. Vi tar bort dessa innan split för att undvika att samma transaktion hamnar i både tränings- och testset (data leakage).

In [2]:
df = pd.read_csv('../data/raw/creditcard.csv')
print(f"Rader innan borttagning av dubbletter: {len(df)}")

df = df.drop_duplicates().reset_index(drop=True)
print(f"Rader efter borttagning av dubbletter: {len(df)}")


Rader innan borttagning av dubbletter: 284807
Rader efter borttagning av dubbletter: 283726


**Vad ser vi?**

1081 rader togs bort (284 807 → 283 726) — exakt matchande de 1081 dubbletterna som identifierades i `01_eda.ipynb`. Bra bekräftelse på att EDA-fyndet och städningen hänger ihop korrekt.

## Separera features och target

In [3]:
X = df.drop('Class', axis=1).copy()
y = df['Class'].copy()

print("X shape:", X.shape)
print("y fördelning:")
print(y.value_counts())


X shape: (283726, 30)
y fördelning:
Class
0    283253
1       473
Name: count, dtype: int64


**Vad ser vi?**

283 726 rader, 30 features (`Time`, `V1`–`V28`, `Amount`). Efter dedup: 283 253 legitima (99,83 %) och **473** bedrägerier — jämfört med de ursprungliga 492 i hela datasetet betyder det att 19 av de borttagna dubbletterna var bedrägerirader. Klassobalansen är i praktiken oförändrad.

## Train/test-split (stratifierad)

Vi delar upp i 80 % train / 20 % test, **stratifierat** på `Class` så att andelen bedrägerier blir densamma i båda delarna. Testsetet rörs inte igen förrän i utvärderingssteget (`04_evaluation.ipynb`).

In [4]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train (full):", X_train_full.shape, "| Bedrägerier:", y_train_full.sum())
print("Test:", X_test.shape, "| Bedrägerier:", y_test.sum())


Train (full): (226980, 30) | Bedrägerier: 378
Test: (56746, 30) | Bedrägerier: 95


**Vad ser vi?**

Train (full): 226 980 rader, 378 bedrägerier → 0,1665 %. Test: 56 746 rader, 95 bedrägerier → 0,1674 %. Andelen bedrägerier är i praktiken identisk i båda delarna, vilket bekräftar att `stratify=y` fungerade som avsett — inget snedvridet urval mellan train och test.

## Skala `Time` och `Amount`

`V1`–`V28` är redan PCA-transformerade och lämnas orörda. `Time` och `Amount` skalas med `StandardScaler`, som **fittas enbart på träningsdatan** (`X_train_full`) för att undvika läckage från testsetet, och sedan används för att transformera båda delarna. Scalern sparas för återanvändning i senare notebooks.

In [5]:
scaler = StandardScaler()

X_train_full_scaled = X_train_full.copy()
X_test_scaled = X_test.copy()

X_train_full_scaled[['Time', 'Amount']] = scaler.fit_transform(X_train_full[['Time', 'Amount']])
X_test_scaled[['Time', 'Amount']] = scaler.transform(X_test[['Time', 'Amount']])

joblib.dump(scaler, '../models/scaler.pkl')
print("Scaler sparad till ../models/scaler.pkl")

X_train_full_scaled[['Time', 'Amount']].describe()


Scaler sparad till ../models/scaler.pkl


,Time,Amount
count,2.269800e+05,2.269800e+05
mean,-1.522636e-16,7.487965e-17
std,1.000002e+00,1.000002e+00
min,-1.998400e+00,-3.596386e-01
25%,-8.561822e-01,-3.364866e-01
50%,-2.123526e-01,-2.697972e-01
75%,9.363191e-01,-4.287453e-02
max,1.640237e+00,7.962087e+01


**Vad ser vi?**

Efter skalning har både `Time` och `Amount` medelvärde ≈ 0 (i storleksordningen 1e-16 — flyttalsbrus, inte en riktig avvikelse) och std = 1,000002 ≈ 1, precis som förväntat för `StandardScaler`. Värt att notera: `Amount`s maxvärde ligger nu på **79,62 standardavvikelser** över medelvärdet — det bekräftar hur extremt högerskevt beloppet var innan skalning (samma mönster vi såg i EDA:ns histogram), och understryker varför skalning behövdes innan träning.

## Extrahera legitima transaktioner för träning + validation-set

Autoencodern ska **bara tränas på legitima transaktioner** (den ska lära sig hur "normalt" ser ut). Vi tar ut dessa ur `X_train_full_scaled` och delar dem ytterligare i en faktisk träningsdel och en validation-del (80/20). Validation-delen används i nästa notebook för att bestämma tröskelvärdet för anomali-flaggning.

In [6]:
X_legit_train_full = X_train_full_scaled[y_train_full == 0].copy()

X_train, X_val = train_test_split(
    X_legit_train_full,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Train (endast legitima):", X_train.shape)
print("Validation (endast legitima):", X_val.shape)


Train (endast legitima): (181281, 30)
Validation (endast legitima): (45321, 30)


**Vad ser vi?**

181 281 rader för träning, 45 321 för validation — en 80/20-split av de 226 602 legitima transaktionerna i träningsdelen (226 980 totalt − 378 bedrägerier). Formerna stämmer matematiskt (181 281 + 45 321 = 226 602).

## Konvertera till NumPy och spara

Vi konverterar till NumPy-arrayer (TensorFlow fungerar bra med båda, men NumPy gör flödet tydligare) och sparar samtliga splits i en enda `.npz`-fil, så att `03_model_training.ipynb` kan läsa in exakt samma data utan att göra om split/skalning.

In [7]:
X_train_np = X_train.to_numpy(dtype=np.float32)
X_val_np = X_val.to_numpy(dtype=np.float32)
X_test_np = X_test_scaled.to_numpy(dtype=np.float32)
y_test_np = y_test.to_numpy()

np.savez(
    '../data/processed/splits.npz',
    X_train=X_train_np,
    X_val=X_val_np,
    X_test=X_test_np,
    y_test=y_test_np
)

print("Sparat till ../data/processed/splits.npz")
print("X_train:", X_train_np.shape)
print("X_val:", X_val_np.shape)
print("X_test:", X_test_np.shape)
print("y_test:", y_test_np.shape, "| Bedrägerier i test:", y_test_np.sum())


Sparat till ../data/processed/splits.npz
X_train: (181281, 30)
X_val: (45321, 30)
X_test: (56746, 30)
y_test: (56746,) | Bedrägerier i test: 95


**Vad ser vi?**

Alla splits sparades med rätt form: `X_train` (181 281, 30), `X_val` (45 321, 30), `X_test` (56 746, 30), `y_test` (56 746,) med 95 bevarade bedrägerier för slutgiltig utvärdering. Antalet kolumner (30) är konsekvent över alla tre set, vilket bekräftar att inga features tappats bort under processen.

## Sammanfattning

- **Dubbletter borttagna:** 284 807 → 283 726 rader (1081 borttagna, varav 19 bedrägerirader)
- **Klassfördelning efter städning:** 283 253 legitima (99,83 %) / 473 bedrägerier (0,17 %) — i praktiken oförändrad obalans
- **Split:** 80/20 train/test, stratifierat — bedrägeriandelen identisk i båda delar (~0,167 %)
- **Skalning:** `Time` och `Amount` standardiserade (fit endast på träningsdata); `Amount`s extrema skevhet syns tydligt i max-värdet (79,6 std)
- **Legitim train/val-split:** 181 281 / 45 321 rader, för autoencoder-träning respektive tröskelbestämning
- **Sparade artefakter:** `../models/scaler.pkl`, `../data/processed/splits.npz` (`X_train`, `X_val`, `X_test`, `y_test`)

**Nästa steg:** `03_model_training.ipynb` — bygg och träna autoencodern på `X_train` (enbart legitima transaktioner), och bestäm tröskelvärdet från rekonstruktionsfelet på `X_val`.